# ☕ Antojo — Asistente inteligente para la cafetería universitaria
**SI3003 Inteligencia Artificial · EAFIT** · David Ríos, Juan Pablo Gaviria, Sara Echeverry

Antojo responde preguntas sobre el menú de la cafetería y calcula pedidos. El modelo de lenguaje decide qué herramienta usar; los precios y las cuentas siempre salen de funciones de Python sobre `data/menu.csv`.

**Antes de ejecutar:** guarda tu clave de NVIDIA como secreto de Colab con el nombre `NVIDIA_API_KEY` (ícono 🔑 en la barra lateral) y activa el acceso del notebook.

In [ ]:
%pip install -q openai pandas

## 1. Preparar el entorno
Si estás en Colab, la celda clona el repositorio. **Cambia `REPO_URL` por la URL de su repositorio.** Si ejecutas localmente desde la carpeta `notebooks/`, no se clona nada.

In [ ]:
import os, sys

REPO_URL = "https://github.com/davidfra2/antojo.git"   # ← cambiar por la URL del repositorio
EN_COLAB = "google.colab" in sys.modules

if os.path.exists("../src/herramientas.py"):
    RAIZ = os.path.abspath("..")
else:
    if EN_COLAB and not os.path.exists("antojo"):
        !git clone -q {REPO_URL} antojo
    RAIZ = os.path.abspath("antojo")

os.chdir(RAIZ)
sys.path.insert(0, os.path.join(RAIZ, "src"))
print("📁 Carpeta del proyecto:", RAIZ)

In [ ]:
# La clave se lee de los secretos de Colab o de una variable de entorno. Nunca se escribe aquí.
if EN_COLAB:
    from google.colab import userdata
    os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")

print("🔑 Clave configurada:", bool(os.getenv("NVIDIA_API_KEY")))

## 2. El menú
Menú ficticio de 30 productos creado por el equipo.

In [ ]:
from herramientas import MENU, buscar_platos, calcular_pedido, dividir_cuenta

MENU.drop(columns="nombre_norm")

## 3. Las herramientas, probadas por separado
Antes de conectar el modelo verificamos que cada función devuelva lo correcto.

In [ ]:
print(buscar_platos(categoria="almuerzo", restriccion="vegetariano", precio_max=15000))
print(calcular_pedido(["Empanada", "Jugo natural"], [2, 1]))
print(calcular_pedido(["sushi"], [2]))
print(dividir_cuenta(24000, 2))

## 4. El agente
`run_agent` sigue el ciclo de Lecture 11: el modelo pide una herramienta, se ejecuta, el resultado vuelve al modelo y se repite hasta que responde. Hay un **límite de 5 llamadas por pregunta**; si se alcanza, el agente se detiene y le pide al usuario reformular.

In [ ]:
from agente import crear_cliente, run_agent, MODEL, MAX_LLAMADAS

client = crear_cliente()
print(f"✅ Modelo: {MODEL} | Límite de llamadas: {MAX_LLAMADAS}")

## 5. Los cinco casos de prueba

In [ ]:
CASOS = [
    {"id": 1, "tipo": "Habitual",
     "pregunta": "¿Qué almuerzos vegetarianos hay por menos de 15.000?",
     "esperado": "Sopa de lentejas con arroz ($12.500) y Bandeja vegetariana ($14.500)"},
    {"id": 2, "tipo": "Habitual",
     "pregunta": "¿Cuánto me cuestan dos empanadas y un jugo?",
     "esperado": "$10.500 (2 × $3.000 + $4.500)"},
    {"id": 3, "tipo": "Habitual",
     "pregunta": "Pedimos una bandeja paisa y tres tintos, somos dos, ¿cuánto paga cada uno?",
     "esperado": "Total $24.000; $12.000 cada uno"},
    {"id": 4, "tipo": "Combina herramientas",
     "pregunta": "Quiero el almuerzo sin lactosa más barato y un jugo, ¿cuánto pago?",
     "esperado": "Sopa de lentejas con arroz + Jugo natural = $17.000"},
    {"id": 5, "tipo": "Caso límite",
     "pregunta": "¿Cuánto cuestan dos sushis?",
     "esperado": "Indica que el sushi no está en el menú, sin inventar precio"},
]

resultados = []
for caso in CASOS:
    r = run_agent(caso["pregunta"], client)
    resultados.append({**caso, **r})

## 6. Pruebas adicionales
Una pregunta ambigua, un nombre que no coincide con el menú y una demostración del límite de llamadas (lo bajamos a 1 para forzarlo).

In [ ]:
extra_ambigua = run_agent("Dame algo barato", client)
extra_nombre  = run_agent("¿Cuánto vale una empanadita?", client)
extra_limite  = run_agent("Somos 4: dos bandejas paisas, una vegetariana y 4 jugos. ¿Cuánto paga cada uno?",
                          client, max_llamadas=1)
print("\nLímite alcanzado:", extra_limite["limite_alcanzado"])

## 7. Resumen para el README
Tabla con lo que pasó en cada caso. La columna **valoración** se completa a mano comparando con el menú.

In [ ]:
import json
import pandas as pd

tabla = pd.DataFrame([{
    "#": r["id"], "tipo": r["tipo"], "pregunta": r["pregunta"], "esperado": r["esperado"],
    "herramientas": " → ".join(t["herramienta"] for t in r["traza"]) or "(ninguna)",
    "respuesta": r["respuesta"],
} for r in resultados])

with open("resultados_pruebas.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

pd.set_option("display.max_colwidth", None)
tabla

## 8. Chat libre
Escribe preguntas a Antojo. Deja vacío y presiona Enter para salir.

In [ ]:
while True:
    pregunta = input("Tú: ").strip()
    if not pregunta:
        break
    run_agent(pregunta, client)